In [7]:
import os
import re
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
# from google.colab import drive

In [16]:
# drive.mount('/content/drive')

DATA_PATH = 'rus.txt'
SAVE_MODEL_PATH = 'best_seq2seq_model.pt'

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


### Data Preprocessing

In [8]:
SOS_TOKEN = '<sos>'
EOS_TOKEN = '<eos>'
PAD_TOKEN = '<pad>'

def clean_text(text):
    """Cleans text by removing punctuation, numbers, and converting to lowercase."""
    text = text.lower()
    text = re.sub(r'[^a-zа-яё\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

class Vocabulary:
    def __init__(self):
        self.word2index = {PAD_TOKEN: 0, SOS_TOKEN: 1, EOS_TOKEN: 2}
        self.index2word = {0: PAD_TOKEN, 1: SOS_TOKEN, 2: EOS_TOKEN}
        self.n_words = 3

    def add_sentence(self, sentence):
        for word in sentence.split(' '):
            self.add_word(word)

    def add_word(self, word):
        if word not in self.word2index and len(word) > 0:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

def load_and_preprocess_data(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        lines = f.read().strip().split('\n')

    eng_vocab = Vocabulary()
    rus_vocab = Vocabulary()
    pairs = []

    for line in lines:
        parts = line.split('\t')
        if len(parts) >= 2:
            eng_sentence = clean_text(parts[0])
            rus_sentence = clean_text(parts[1])

            if eng_sentence and rus_sentence:
                eng_vocab.add_sentence(eng_sentence)
                rus_vocab.add_sentence(rus_sentence)
                pairs.append((eng_sentence, rus_sentence))

    random.shuffle(pairs)
    split_idx = int(len(pairs) * 0.8)
    train_pairs = pairs[:split_idx]
    val_pairs = pairs[split_idx:]

    return eng_vocab, rus_vocab, train_pairs, val_pairs

eng_vocab, rus_vocab, train_pairs, val_pairs = load_and_preprocess_data(DATA_PATH)
print(f"Total train pairs: {len(train_pairs)} | Total val pairs: {len(val_pairs)}")

Total train pairs: 290708 | Total val pairs: 72678


### Dataset, Dataloader

In [ ]:
class TranslationDataset(Dataset):
    def __init__(self, pairs, eng_vocab, rus_vocab):
        self.pairs = pairs
        self.eng_vocab = eng_vocab
        self.rus_vocab = rus_vocab

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        eng_sentence, rus_sentence = self.pairs[idx]

        eng_indices = [self.eng_vocab.word2index[w] for w in eng_sentence.split(' ')]
        rus_indices = [self.rus_vocab.word2index[SOS_TOKEN]] + \
                      [self.rus_vocab.word2index[w] for w in rus_sentence.split(' ')] + \
                      [self.rus_vocab.word2index[EOS_TOKEN]]

        return torch.tensor(eng_indices, dtype=torch.long), torch.tensor(rus_indices, dtype=torch.long)

def collate_fn(batch):
    eng_batch, rus_batch = zip(*batch)
    eng_padded = pad_sequence(eng_batch, padding_value=0, batch_first=False)
    rus_padded = pad_sequence(rus_batch, padding_value=0, batch_first=False)
    return eng_padded, rus_padded

BATCH_SIZE = 64
train_dataset = TranslationDataset(train_pairs, eng_vocab, rus_vocab)
val_dataset = TranslationDataset(val_pairs, eng_vocab, rus_vocab)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

### Model architecture

In [14]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)
        return hidden, cell

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hidden_dim):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        input_token = input_token.unsqueeze(0) # (1, batch_size)
        embedded = self.embedding(input_token)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        prediction = self.fc_out(output.squeeze(0)) # (batch_size, output_dim)
        return prediction, hidden, cell

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        # src: (src_len, batch_size)
        # trg: (trg_len, batch_size)
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(trg_len, batch_size, trg_vocab_size).to(self.device)
        hidden, cell = self.encoder(src)

        # First input to the decoder is the <sos> token
        input_token = trg[0, :]

        # Generate sequence one token at a time
        for t in range(1, trg_len):
            output, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[t] = output

            top1 = output.argmax(1)
            teacher_force = random.random() < teacher_forcing_ratio
            input_token = trg[t] if teacher_force else top1

        return outputs

INPUT_DIM = eng_vocab.n_words
OUTPUT_DIM = rus_vocab.n_words
ENC_EMB_DIM = 256
DEC_EMB_DIM = 256
HID_DIM = 1024

encoder = Encoder(INPUT_DIM, ENC_EMB_DIM, HID_DIM)
decoder = Decoder(OUTPUT_DIM, DEC_EMB_DIM, HID_DIM)
model = Seq2Seq(encoder, decoder, device).to(device)


### Training loop

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=0)

def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0

    for i, (src, trg) in enumerate(iterator):
        src, trg = src.to(device), trg.to(device)
        optimizer.zero_grad()

        output = model(src, trg)
        # trg: (trg_len, batch_size) -> output: (trg_len, batch_size, output_dim)
        output_dim = output.shape[-1]
        output = output[1:].view(-1, output_dim)
        trg = trg[1:].view(-1)
        loss = criterion(output, trg)
        loss.backward()

        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(iterator)

def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for i, (src, trg) in enumerate(iterator):
            src, trg = src.to(device), trg.to(device)

            output = model(src, trg, teacher_forcing_ratio=0)

            output_dim = output.shape[-1]
            output = output[1:].view(-1, output_dim)
            trg = trg[1:].view(-1)

            loss = criterion(output, trg)
            epoch_loss += loss.item()

    return epoch_loss / len(iterator)

N_EPOCHS = 10
CLIP = 1.0
best_valid_loss = float('inf')

print("Starting Training...")
for epoch in range(N_EPOCHS):
    train_loss = train(model, train_loader, optimizer, criterion, CLIP)
    valid_loss = evaluate(model, val_loader, criterion)

    # Save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), SAVE_MODEL_PATH)
        print(f"Epoch: {epoch+1:02} | Model saved! (Val Loss decreased)")

    print(f'Epoch: {epoch+1:02} | Train Loss: {train_loss:.3f} | Val. Loss: {valid_loss:.3f}')

Starting Training...
Epoch: 01 | Model saved! (Val Loss decreased)
Epoch: 01 | Train Loss: 3.488 | Val. Loss: 2.802
Epoch: 02 | Model saved! (Val Loss decreased)
Epoch: 02 | Train Loss: 1.910 | Val. Loss: 2.493
Epoch: 03 | Model saved! (Val Loss decreased)
Epoch: 03 | Train Loss: 1.404 | Val. Loss: 2.460
Epoch: 04 | Train Loss: 1.139 | Val. Loss: 2.468
Epoch: 05 | Train Loss: 1.002 | Val. Loss: 2.514
Epoch: 06 | Train Loss: 0.903 | Val. Loss: 2.548


KeyboardInterrupt: 

translation

In [11]:
def translate_sentence(sentence, src_vocab, trg_vocab, model, device, max_len=50):
    model.eval()

    # Handle both string input and tensor input
    if isinstance(sentence, str):
        cleaned_sentence = clean_text(sentence)
        tokens = cleaned_sentence.split(' ')
        indices = [src_vocab.word2index.get(token, 0) for token in tokens] # 0 for unk/pad
        tensor = torch.LongTensor(indices).unsqueeze(1).to(device)
    else:
        tensor = sentence.unsqueeze(1).to(device) if sentence.dim() == 1 else sentence.to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(tensor)

    trg_indexes = [trg_vocab.word2index[SOS_TOKEN]]

    for i in range(max_len):
        trg_tensor = torch.LongTensor([trg_indexes[-1]]).to(device)

        with torch.no_grad():
            output, hidden, cell = model.decoder(trg_tensor, hidden, cell)

        pred_token = output.argmax(1).item()
        trg_indexes.append(pred_token)

        # Stop translating if we reach <eos>
        if pred_token == trg_vocab.word2index[EOS_TOKEN]:
            break

    trg_tokens = [trg_vocab.index2word[i] for i in trg_indexes]

    # Remove <sos> and <eos> tokens
    return trg_tokens[1:-1]

In [18]:
# Testing the translation function
model.load_state_dict(torch.load(SAVE_MODEL_PATH, map_location=torch.device('cpu')))
sample_english = "this is a very long sentence"
translated = translate_sentence(sample_english, eng_vocab, rus_vocab, model, device)
print(f"\nEnglish: {sample_english}")
print(f"Russian translation: {' '.join(translated)}")


English: this is a very long sentence
Russian translation: это очень длинная предложение


## homework 6: bleu and beam search

In [20]:
import math
import time
import random
import pandas as pd
from collections import Counter
import torch
import torch.nn.functional as F

bleu eval

In [2]:
def get_ngrams(tokens, n):
    return Counter(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def clipped_precision(hypothesis, reference, n):
    hyp_ngrams = get_ngrams(hypothesis, n)
    ref_ngrams = get_ngrams(reference, n)

    if len(hyp_ngrams) == 0: return 0.0

    clipped_counts = sum(min(count, ref_ngrams.get(ngram, 0)) for ngram, count in hyp_ngrams.items())
    return clipped_counts / sum(hyp_ngrams.values())

def brevity_penalty(hypothesis, reference):
    hyp_len, ref_len = len(hypothesis), len(reference)
    if hyp_len == 0: return 0.0
    return 1.0 if hyp_len > ref_len else math.exp(1 - ref_len / hyp_len)

def bleu_score(hypothesis, reference):
    log_precisions = 0.0
    for n in range(1, 5):
        p_n = clipped_precision(hypothesis, reference, n)
        if p_n == 0: return 0.0
        log_precisions += 0.25 * math.log(p_n)

    return brevity_penalty(hypothesis, reference) * math.exp(log_precisions)


beam search

In [21]:
def beam_search(model, sentence, beam_width, src_vocab=eng_vocab, trg_vocab=rus_vocab, device=device, max_len=50):
    model.eval()

    # Process source sentence
    if isinstance(sentence, str):
        tokens = [tok.lower() for tok in re.sub(r"[^a-zA-Zа-яА-ЯёЁ]+", " ", sentence).strip().split()]
        indices = [src_vocab.word2index.get(tok, 0) for tok in tokens]
        tensor = torch.LongTensor(indices).unsqueeze(1).to(device)
    else:
        tensor = sentence.unsqueeze(1).to(device) if sentence.dim() == 1 else sentence.to(device)

    with torch.no_grad():
        hidden, cell = model.encoder(tensor)

    # Beam tracks: (cumulative_log_prob, [token_sequence], hidden_state, cell_state)
    beam = [(0.0, [trg_vocab.word2index[SOS_TOKEN]], hidden, cell)]
    completed = []

    for _ in range(max_len):
        new_beam = []
        for score, seq, h, c in beam:
            if seq[-1] == trg_vocab.word2index[EOS_TOKEN]:
                completed.append((score, seq))
                continue

            trg_tensor = torch.LongTensor([seq[-1]]).to(device)
            with torch.no_grad():
                output, new_h, new_c = model.decoder(trg_tensor, h, c)
                log_probs = F.log_softmax(output.squeeze(0), dim=-1)

            topk_log_probs, topk_indices = log_probs.topk(beam_width)
            for i in range(beam_width):
                new_beam.append((score + topk_log_probs[i].item(), seq + [topk_indices[i].item()], new_h, new_c))

        if not new_beam: break
        beam = sorted(new_beam, key=lambda x: x[0], reverse=True)[:beam_width]

    if not completed:
        completed = [(score, seq) for score, seq, _, _ in beam]

    best_seq = sorted(completed, key=lambda x: x[0], reverse=True)[0][1]

    # Convert to words and remove <sos>/<eos>
    trg_tokens = [trg_vocab.index2word[i] for i in best_seq]
    return [t for t in trg_tokens if t not in [SOS_TOKEN, EOS_TOKEN]]

comparison

In [22]:
test_sample = random.sample(val_pairs, 100)
results = []

for config, width in [("Greedy", None), ("Beam (w=3)", 3), ("Beam (w=5)", 5), ("Beam (w=10)", 10)]:
    total_time, total_bleu = 0, 0

    for eng, rus in test_sample:
        ref_tokens = rus.split()

        start = time.time()
        if width is None:
            hyp_tokens = translate_sentence(eng, eng_vocab, rus_vocab, model, device)
        else:
            hyp_tokens = beam_search(model, eng, width)
        total_time += (time.time() - start)

        total_bleu += bleu_score(hyp_tokens, ref_tokens)

    results.append({
        "Configuration": config,
        "BLEU-4": total_bleu / 100,
        "Avg Time (s)": total_time / 100
    })

print("\n--- EXPERIMENT RESULTS ---")
display(pd.DataFrame(results))



--- EXPERIMENT RESULTS ---


,Configuration,BLEU-4,Avg Time (s)
0,Greedy,0.237530,0.072367
1,Beam (w=3),0.262506,0.461999
2,Beam (w=5),0.267461,1.142643
3,Beam (w=10),0.267461,3.145946


In [23]:
print("\n--- 5 TRANSLATION EXAMPLES ---")
for i, (eng, rus) in enumerate(random.sample(val_pairs, 5)):
    greedy_out = translate_sentence(eng, eng_vocab, rus_vocab, model, device)
    beam_out = beam_search(model, eng, beam_width=5)

    print(f"\nExample {i+1}:")
    print(f"Source (EN):    {eng}")
    print(f"Reference (RU): {rus}")
    print(f"Greedy Output:  {' '.join(greedy_out)}")
    print(f"Beam (w=5):     {' '.join(beam_out)}")


--- 5 TRANSLATION EXAMPLES ---

Example 1:
Source (EN):    does it really hurt
Reference (RU): это действительно больно
Greedy Output:  это действительно больно
Beam (w=5):     это действительно больно

Example 2:
Source (EN):    tom only eats vegetables that he grows himself
Reference (RU): том ест только те овощи которые выращивает сам
Greedy Output:  том ест только ест овощи он выращивает
Beam (w=5):     том ест только ест овощи

Example 3:
Source (EN):    there isnt much furniture in my room
Reference (RU): у меня в комнате не очень много мебели
Greedy Output:  в моей комнате много мебели
Beam (w=5):     в моей комнате много мебели

Example 4:
Source (EN):    we bought a round table
Reference (RU): мы купили круглый стол
Greedy Output:  мы купили круглый стол
Beam (w=5):     мы купили круглый стол

Example 5:
Source (EN):    this exam is very easy
Reference (RU): это очень простой экзамен
Greedy Output:  это очень легко лёгким
Beam (w=5):     это очень очень легко
